# HyDRA v6 — Live Visualizer
**Modo:** v6 (GCM Full) | v5 | v4 | v3 | Magnetic

> Notebook paralelo — CPU runtime, sem GPU.
> Visualiza espaço hiperbólico, temporal binding, Persistence Landscape, DGW e fase do protocolo 3-fases.

In [1]:
"""Cell 1 — Mount + Config"""
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
from pathlib import Path
import os

RUN_MODE = "v6"  # "v6" | "v5" | "v4" | "v3" | "magnetic"

ROOTS = {
    "magnetic": Path("/content/drive/MyDrive/HydraPaper_Magnetic"),
    "v3":       Path("/content/drive/MyDrive/HydraPaper_v3"),
    "v4":       Path("/content/drive/MyDrive/HydraPaper_v4"),
    "v5":       Path("/content/drive/MyDrive/HydraPaper_v5"),
    "v6":       Path("/content/drive/MyDrive/HydraPaper_v6"),
}
RUN_TAGS = {
    "magnetic": "hydra_magnetic_v1",
    "v3":       "hydra_v3_temporal_binding",
    "v4":       "hydra_v4_cgt_binding",
    "v5":       "hydra_v5_anosov_topo",
    "v6":       "hydra_v6_gcm",
}
LABELS = {
    "magnetic": "HyDRA Magnetic",
    "v3":       "HyDRA v3 · Temporal Binding",
    "v4":       "HyDRA v4 · CGT + Binding",
    "v5":       "HyDRA v5 · Anosov Topo",
    "v6":       "HyDRA v6 · GCM Full Architecture",
}
PHASE_LABELS = { "v6": {0:"ALIGN",1:"REFINE",2:"GEO"} }
PHASE_COLORS = { 0:"#4f8fff", 1:"#ffb84f", 2:"#ff6bff" }
PHASE_ENDS   = { "v6": (3000, 8000) }

PAPER_ROOT    = ROOTS[RUN_MODE]
RUN_TAG       = RUN_TAGS[RUN_MODE]
LOG_DIR       = PAPER_ROOT / "logs" / RUN_TAG
TRAIN_CSV     = LOG_DIR / "training_metrics.csv"
VAL_CSV       = LOG_DIR / "val_metrics.csv"
BINDING_CSV   = LOG_DIR / "binding_metrics.csv"
SNAPSHOT_PATH = LOG_DIR / "embeddings_snapshot.json"
GW_CSV        = LOG_DIR / "gw_metrics.csv"
TIMELAPSE_DIR = PAPER_ROOT / "timelapse"
TIMELAPSE_DIR.mkdir(parents=True, exist_ok=True)
RUN_LABEL     = LABELS[RUN_MODE]

print(f"Mode    : {RUN_MODE.upper()} — {RUN_LABEL}")
print(f"Log dir : {LOG_DIR}")
for name, path in [("Train",TRAIN_CSV),("Val",VAL_CSV),
                    ("Binding",BINDING_CSV),("Snapshot",SNAPSHOT_PATH),
                    ("GW",GW_CSV)]:
    print(f"  {name:<8}: {'✅' if path.exists() else '⏳'}")


Mounted at /content/drive
Mode    : V6 — HyDRA v6 · GCM Full Architecture
Log dir : /content/drive/MyDrive/HydraPaper_v6/logs/hydra_v6_gcm
  Train   : ⏳
  Val     : ⏳
  Binding : ⏳
  Snapshot: ⏳
  GW      : ⏳


In [2]:
"""Cell 2 — Utilities"""
import csv, json as _json, math, time, base64
from pathlib import Path

def read_csv(p):
    try: return list(csv.DictReader(open(p)))
    except: return []

def latest_metrics(train, val):
    t = train[-1] if train else {}
    v = val[-1]   if val   else {}
    return {
        "step":    int(t.get("step",0) or 0),
        "loss":    float(t.get("loss",0) or 0),
        "ppl":     float(t.get("ppl",9999) or 9999),
        "rdc":     float(t.get("rdc_ema",0) or 0),
        "rad":     float(t.get("mean_radius",1.5) or 1.5),
        "div":     float(t.get("diversity",0) or 0),
        "hid":     float(t.get("l_hidden",0) or 0),
        "lr":      float(t.get("lr",0) or 0),
        "logit_std": float(t.get("logit_std",0) or 0),
        "w_ent":   float(t.get("w_entropy",0) or 0),
        "val_ppl": float(v.get("val_ppl",9999) or 9999),
        "val_step":int(v.get("step",0) or 0),
    }

def latest_binding(rows):
    b = rows[-1] if rows else {}
    try:    phases = _json.loads(b.get("phases_json","[]"))
    except: phases = []
    return {
        "order_param":  float(b.get("order_param",0) or 0),
        "n_groups":     int(float(b.get("n_groups",0) or 0)),
        "binding_loss": float(b.get("binding_loss",0) or 0),
        "phases":       phases,
    }

def load_snapshot():
    if not SNAPSHOT_PATH.exists(): return {}
    try: return _json.loads(SNAPSHOT_PATH.read_text())
    except: return {}

def get_training_phase(step):
    """Detect current training phase for v6."""
    if RUN_MODE != "v6": return None
    ph1, ph2 = PHASE_ENDS.get(RUN_MODE, (3000, 8000))
    if step < ph1:   return 0
    if step < ph2:   return 1
    return 2

print("✅ Utilities ready")


✅ Utilities ready


In [3]:
"""Cell 3 — Live Metrics Charts (with Phase indicator + DGW)"""
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

def render_charts():
    train   = read_csv(TRAIN_CSV)
    val     = read_csv(VAL_CSV)
    binding = read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6") else []
    gw_rows = read_csv(GW_CSV)      if RUN_MODE == "v6" else []
    if not train: print("⏳ Aguardando dados..."); return

    m  = latest_metrics(train, val)
    bm = latest_binding(binding)

    steps  = [int(r["step"])           for r in train]
    ppls   = [float(r["ppl"])          for r in train if r.get("ppl") and float(r["ppl"])<5000]
    losses = [float(r["loss"])         for r in train if r.get("loss")]
    rdcs   = [float(r["rdc_ema"])      for r in train if r.get("rdc_ema")]
    rads   = [float(r["mean_radius"])  for r in train if r.get("mean_radius")]
    divs   = [float(r["diversity"])    for r in train if r.get("diversity")]
    hids   = [float(r["l_hidden"])     for r in train if r.get("l_hidden")]
    lstds  = [float(r["logit_std"])    for r in train if r.get("logit_std")]
    w_ents = [float(r["w_entropy"])    for r in train if r.get("w_entropy")]
    vs     = [int(r["step"])           for r in val]
    vp     = [float(r["val_ppl"])      for r in val if r.get("val_ppl")]
    b_rows = binding
    b_s    = [int(r["step"])                   for r in b_rows]
    b_g    = [float(r.get("order_param",0) or 0) for r in b_rows]
    b_l    = [float(r.get("binding_loss",0) or 0) for r in b_rows]
    gw_s   = [int(r["step"])          for r in gw_rows]
    gw_d   = [float(r.get("dgw",0) or 0) for r in gw_rows]

    is_v6      = RUN_MODE == "v6"
    has_bind   = RUN_MODE in ("v3","v4","v5","v6")
    n_rows     = 5 if is_v6 else (4 if has_bind else 3)
    titles     = (["PPL","Loss","RDC","Radius","logit_std","w_entropy"] +
                  (["Γ Order Param","Binding Loss"] if has_bind else []) +
                  (["DGW Divergence","Phase"] if is_v6 else []))

    fig = make_subplots(rows=n_rows, cols=2,
        subplot_titles=titles[:n_rows*2],
        vertical_spacing=0.09, horizontal_spacing=0.07)

    kw = dict(mode="lines", showlegend=False)
    # Row 1: PPL + Loss
    fig.add_trace(go.Scatter(x=steps[:len(ppls)], y=ppls,
        line=dict(color="#4f8fff",width=1.5), name="ppl", **kw), row=1,col=1)
    if vp: fig.add_trace(go.Scatter(x=vs[:len(vp)], y=vp,
        mode="lines+markers", line=dict(color="#4fffa0",width=2),
        marker=dict(size=4), showlegend=False), row=1,col=1)
    fig.add_trace(go.Scatter(x=steps[:len(losses)], y=losses,
        line=dict(color="#b07fff",width=1.5), **kw), row=1,col=2)

    # Row 2: RDC + Radius
    fig.add_trace(go.Scatter(x=steps[:len(rdcs)], y=rdcs,
        line=dict(color="#ff6b6b",width=1.5), **kw), row=2,col=1)
    fig.add_hline(y=2.0, line_dash="dash", line_color="#ff6b6b", row=2, col=1)
    fig.add_trace(go.Scatter(x=steps[:len(rads)], y=rads,
        line=dict(color="#ffb84f",width=1.5), **kw), row=2,col=2)
    fig.add_hline(y=1.5, line_dash="dash", line_color="#ffb84f", row=2, col=2)

    # Row 3: logit_std + w_entropy
    fig.add_trace(go.Scatter(x=steps[:len(lstds)], y=lstds,
        line=dict(color="#7ec8ff",width=1.5), **kw), row=3,col=1)
    fig.add_trace(go.Scatter(x=steps[:len(w_ents)], y=w_ents,
        line=dict(color="#4fffa0",width=1.5), **kw), row=3,col=2)

    # Row 4: Binding (v3+)
    if has_bind and b_s:
        fig.add_trace(go.Scatter(x=b_s, y=b_g,
            line=dict(color="#ff6bff",width=2), **kw), row=4,col=1)
        fig.add_hline(y=0.3, line_dash="dash", line_color="#ff6bff", row=4, col=1)
        fig.add_trace(go.Scatter(x=b_s, y=b_l,
            line=dict(color="#cc44cc",width=1.5), **kw), row=4,col=2)

    # Row 5: DGW + Phase protocol (v6 only)
    if is_v6:
        if gw_s:
            fig.add_trace(go.Scatter(x=gw_s, y=gw_d,
                line=dict(color="#ff9f43",width=2), **kw), row=5,col=1)
        # Phase timeline bar
        ph1, ph2 = PHASE_ENDS.get(RUN_MODE, (3000,8000))
        max_step = max(steps[-1] if steps else 0, ph2+1000)
        for ph, (s, e, col, lbl) in enumerate([
            (0,    ph1, "#4f8fff", "P1 ALIGN"),
            (ph1,  ph2, "#ffb84f", "P2 REFINE"),
            (ph2,  max_step, "#ff6bff", "P3 GEO"),
        ]):
            fig.add_trace(go.Bar(x=[lbl], y=[1],
                marker_color=col, opacity=0.7,
                showlegend=False, width=0.6), row=5,col=2)
        # Current phase marker
        cur_ph = get_training_phase(m["step"])
        if cur_ph is not None:
            ph_lbl = PHASE_LABELS["v6"][cur_ph]
            fig.add_annotation(
                text=f"▶ {ph_lbl}", xref="paper", yref="paper",
                x=0.99, y=0.02, font=dict(color=PHASE_COLORS[cur_ph], size=11),
                showarrow=False)

    # Phase shading on PPL chart for v6
    if is_v6 and steps:
        ph1, ph2 = PHASE_ENDS.get(RUN_MODE, (3000,8000))
        for s, e, col in [(0, ph1, "#4f8fff"), (ph1, ph2, "#ffb84f"), (ph2, 200000, "#ff6bff")]:
            fig.add_vrect(x0=s, x1=min(e, steps[-1]+500),
                fillcolor=col, opacity=0.04, layer="below",
                line_width=0, row=1, col=1)

    cur_ph   = get_training_phase(m["step"])
    ph_str   = f" · Phase {cur_ph+1} {PHASE_LABELS['v6'].get(cur_ph,'')}" if is_v6 and cur_ph is not None else ""
    bind_str = f" · Γ={bm['order_param']:.3f}" if has_bind and bm else ""
    gw_str   = f" · DGW={gw_d[-1]:.4f}" if gw_d else ""

    fig.update_layout(
        height=1050 if is_v6 else (850 if has_bind else 680),
        paper_bgcolor="#05080f", plot_bgcolor="#0d1120",
        font=dict(color="#c8d4f0", size=10), showlegend=False,
        title=dict(
            text=(f"{RUN_LABEL} · Step {m['step']:,}/200k · "
                  f"PPL {m['ppl']:.1f} · rdc {m['rdc']:.3f}"
                  f"{ph_str}{bind_str}{gw_str}"),
            font=dict(size=11, color="#fff")),
        margin=dict(t=50, b=15, l=40, r=20))
    for i in range(1, n_rows+1):
        for j in range(1, 3):
            fig.update_xaxes(gridcolor="#1a2035", row=i, col=j)
            fig.update_yaxes(gridcolor="#1a2035", row=i, col=j)
    fig.show()

render_charts()


⏳ Aguardando dados...


In [4]:
"""Cell 4 — Kuramoto Phase Portrait + Γ History"""
import plotly.graph_objects as go
import math, cmath

def render_phase_portrait():
    if RUN_MODE not in ("v3","v4","v5","v6"):
        print("ℹ️  Phase portrait: v3/v4/v5/v6 only"); return

    snap    = load_snapshot()
    b_rows  = read_csv(BINDING_CSV)
    bm      = latest_binding(b_rows)
    phases  = snap.get("phases", bm.get("phases", []))

    if not phases and not b_rows:
        print("⏳ Aguardando snapshot de binding..."); return

    # ── Polar plot ────────────────────────────────────────────────────────
    fig = go.Figure()
    if phases:
        import colorsys
        for h, ph in enumerate(phases):
            hue = h / max(len(phases), 1)
            r2, g2, b2 = [int(x*255) for x in colorsys.hsv_to_rgb(hue*0.7+0.55, 0.9, 0.9)]
            col = f"rgb({r2},{g2},{b2})"
            fig.add_trace(go.Scatterpolar(
                r=[0, 1.0], theta=[0, math.degrees(ph)],
                mode="lines+markers",
                line=dict(color=col, width=2.5),
                marker=dict(size=[0, 9], color=col),
                name=f"θ{h}"))

        gv   = sum(cmath.exp(1j*p) for p in phases) / len(phases)
        gm   = abs(gv)
        gang = cmath.phase(gv)
        fig.add_trace(go.Scatterpolar(
            r=[0, gm], theta=[0, math.degrees(gang)],
            mode="lines+markers",
            line=dict(color="#ff6bff", width=5, dash="dash"),
            marker=dict(size=[0, 16], color="#ff6bff", symbol="star"),
            name=f"Γ={gm:.3f}"))
        fig.add_trace(go.Scatterpolar(
            r=[1.0]*361, theta=list(range(361)),
            mode="lines", line=dict(color="#1a2540", width=1, dash="dot"),
            showlegend=False))

    # Phase indicator for v6
    cur_ph = get_training_phase(snap.get("step", 0))
    ph_str = ""
    if RUN_MODE == "v6" and cur_ph is not None:
        ph_str = f" · {PHASE_LABELS['v6'][cur_ph]}"

    fig.update_layout(
        polar=dict(
            radialaxis=dict(range=[0,1.15], gridcolor="#1a2540",
                           color="#3a4a6a", tickfont=dict(size=8)),
            angularaxis=dict(gridcolor="#1a2540", color="#3a4a6a"),
            bgcolor="#0d1120"),
        paper_bgcolor="#05080f",
        font=dict(color="#c8d4f0", family="'DM Mono', monospace", size=10),
        title=dict(
            text=(f"Kuramoto Phase Portrait · Γ={bm['order_param']:.3f} · "
                  f"{len(phases)} osciladores{ph_str}"),
            font=dict(size=11, color="#fff")),
        height=480, showlegend=True,
        legend=dict(bgcolor="#05080f", bordercolor="#1a2540", font=dict(size=8)),
        margin=dict(t=50, b=15, l=15, r=15))
    fig.show()

    # Γ history + DGW overlay
    if b_rows:
        b_s  = [int(r["step"])                   for r in b_rows]
        b_g  = [float(r.get("order_param",0) or 0) for r in b_rows]
        b_l  = [float(r.get("binding_loss",0) or 0) for r in b_rows]
        gw_rows = read_csv(GW_CSV) if RUN_MODE == "v6" else []
        gw_s = [int(r["step"])        for r in gw_rows]
        gw_d = [float(r.get("dgw",0)) for r in gw_rows]

        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(x=b_s, y=b_g, mode="lines",
            line=dict(color="#ff6bff",width=2), name="Γ order param"))
        fig2.add_hline(y=0.3, line_dash="dash", line_color="#ff6bff",
            annotation_text="Γ_min=0.3")
        if gw_d:
            fig2.add_trace(go.Scatter(x=gw_s, y=gw_d, mode="lines+markers",
                line=dict(color="#ff9f43",width=2), marker=dict(size=6),
                name="DGW(t)", yaxis="y2"))

        # Phase shading
        if RUN_MODE == "v6" and b_s:
            ph1, ph2 = PHASE_ENDS.get("v6", (3000,8000))
            for s, e, col, lbl in [
                (0, ph1, "#4f8fff","P1"),(ph1,ph2,"#ffb84f","P2"),
                (ph2, max(b_s[-1],ph2+100), "#ff6bff","P3")]:
                fig2.add_vrect(x0=s, x1=min(e, b_s[-1]+500),
                    fillcolor=col, opacity=0.07, layer="below", line_width=0)
                fig2.add_annotation(x=(s+min(e,b_s[-1]))/2, y=0.95,
                    text=lbl, font=dict(color=col, size=9),
                    xref="x", yref="paper", showarrow=False)

        fig2.update_layout(
            paper_bgcolor="#05080f", plot_bgcolor="#0d1120",
            font=dict(color="#c8d4f0", size=10),
            title=dict(text="Γ + DGW(t) over Training" + (" · Phase Shading" if RUN_MODE=="v6" else ""),
                      font=dict(size=11, color="#fff")),
            height=280,
            yaxis=dict(title="Γ", gridcolor="#1a2035", range=[0,1.1]),
            yaxis2=dict(title="DGW", overlaying="y", side="right",
                       gridcolor="#1a2035", showgrid=False),
            legend=dict(bgcolor="#05080f"),
            margin=dict(t=45, b=20, l=50, r=60))
        fig2.show()

render_phase_portrait()


⏳ Aguardando snapshot de binding...


In [5]:
"""Cell 5 — 3D Hyperbolic World (GCM Full Architecture)"""
import json as _json
from IPython.display import HTML

def build_hyperboloid_viz():
    train   = read_csv(TRAIN_CSV)
    val     = read_csv(VAL_CSV)
    binding = read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6") else []
    gw_rows = read_csv(GW_CSV)      if RUN_MODE == "v6" else []
    m       = latest_metrics(train, val) if train else {}
    bm      = latest_binding(binding)
    snap    = load_snapshot()

    has_snap    = bool(snap.get("tokens"))
    tokens_json = _json.dumps(snap.get("tokens", []))
    phases_json = _json.dumps(snap.get("phases", bm.get("phases", [])))
    trail       = train[-600:] if len(train) > 600 else train
    trail_json  = _json.dumps([{
        "step": int(r.get("step",0) or 0),
        "ppl":  min(float(r.get("ppl",999) or 999), 3000),
        "rdc":  float(r.get("rdc_ema",0) or 0),
    } for r in trail])
    gw_json = _json.dumps([{
        "step": int(r["step"]),
        "dgw":  float(r.get("dgw",0) or 0),
    } for r in gw_rows])

    cur_ph    = get_training_phase(m.get("step", 0))
    ph_name   = PHASE_LABELS.get("v6", {}).get(cur_ph, "") if RUN_MODE=="v6" else ""
    ph_color  = PHASE_COLORS.get(cur_ph, "#ffffff") if cur_ph is not None else "#ffffff"
    gamma     = bm.get("order_param", 0.0)
    cur_rdc   = m.get("rdc", 0.2)
    cur_rad   = m.get("rad", 1.5)
    cur_logstd= m.get("logit_std", 0.0)
    cur_went  = m.get("w_ent", 5.5)
    is_v3v4v5v6 = "true" if RUN_MODE in ("v3","v4","v5","v6") else "false"
    is_v4v5v6   = "true" if RUN_MODE in ("v4","v5","v6") else "false"
    is_v6_str   = "true" if RUN_MODE == "v6" else "false"
    snap_status = f"✅ Snapshot @ step {snap.get('step',0):,}" if has_snap else "⏳ Aguardando snapshot"
    dgw_last    = gw_rows[-1]["dgw"] if gw_rows else 0.0

    html = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:wght@400;500&family=Bebas+Neue&display=swap');
:root {{
  --bg:     #02030a;
  --panel:  rgba(6,10,22,0.92);
  --accent: #4f8fff;
  --bind:   #ff6bff;
  --cgt:    #7ec8ff;
  --topo:   #4fffa0;
  --gw:     #ff9f43;
  --warn:   #ffb84f;
  --dim:    #2a3550;
  --text:   #c8d4f0;
}}
#hw-root {{ background:var(--bg); border-radius:10px; overflow:hidden;
           position:relative; width:100%; font-family:"DM Mono",monospace; }}
canvas#hw-c {{ width:100%;height:100%;display:block; }}

/* HUD left */
.hw-hud {{
  position:absolute; top:0; left:0; padding:16px;
  pointer-events:none; z-index:10; line-height:1.75;
  font-size:10px; color:var(--text);
  background:linear-gradient(135deg,rgba(2,3,10,0.85) 0%,transparent 100%);
  border-radius:10px 0 0 0;
}}
.hw-title {{
  font-family:"Bebas Neue",sans-serif; font-size:18px;
  letter-spacing:.12em; color:#fff; margin-bottom:8px; display:block;
}}
.hw-phase {{
  font-family:"Bebas Neue",sans-serif; font-size:13px;
  letter-spacing:.1em; color:{ph_color}; margin-bottom:6px;
}}
.hw-ok  {{ color:var(--topo); }}
.hw-warn{{ color:var(--warn); }}
.hw-bind{{ color:var(--bind); }}
.hw-cgt {{ color:var(--cgt);  }}
.hw-gw  {{ color:var(--gw);   }}
.hw-dim {{ color:var(--dim);  }}

/* HUD right */
.hw-snap {{
  position:absolute; top:14px; right:14px; font-size:8.5px;
  color:#3a4a6a; z-index:10; text-align:right; pointer-events:none;
}}

/* Legend bottom-left */
.hw-legend {{
  position:absolute; bottom:14px; left:14px; font-size:8px;
  color:#2a3550; pointer-events:none; z-index:10; line-height:2.0;
}}

/* Gamma badge bottom-right */
.hw-gamma-badge {{
  position:absolute; bottom:14px; right:14px;
  font-family:"Bebas Neue",sans-serif; font-size:28px;
  color:var(--bind); opacity:0.75; pointer-events:none; z-index:10;
  text-shadow: 0 0 20px var(--bind);
}}
/* DGW badge */
.hw-dgw-badge {{
  position:absolute; bottom:52px; right:14px;
  font-family:"Bebas Neue",sans-serif; font-size:16px;
  color:var(--gw); opacity:0.8; pointer-events:none; z-index:10;
}}
</style>

<div id="hw-root" style="height:720px;">
<canvas id="hw-c"></canvas>

<div class="hw-hud">
  <span class="hw-title">{RUN_LABEL[:24]}</span>
  {"<span class='hw-phase'>⬡ PHASE " + str((cur_ph or 0)+1) + " — " + ph_name + "</span>" if RUN_MODE=="v6" and cur_ph is not None else ""}
  <div>STEP &nbsp;&nbsp;{m.get("step",0):>8,} / 200,000</div>
  <div>PPL  &nbsp;&nbsp;{m.get("ppl",9999):>8.1f}</div>
  <div class="hw-ok">RDC  &nbsp;&nbsp;{m.get("rdc",0):>8.3f} ✓</div>
  <div class="hw-warn">RAD  &nbsp;&nbsp;{m.get("rad",1.5):>8.3f}</div>
  <div>LSTD &nbsp;&nbsp;{cur_logstd:>8.3f}</div>
  <div>W_ENT&nbsp;&nbsp;{cur_went:>8.3f}</div>
  <div class="hw-dim" style="margin-top:4px;">val PPL {m.get("val_ppl",9999):>6.1f}@{m.get("val_step",0)}</div>
  {"<div class='hw-bind'>Γ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;" + f"{gamma:>8.3f}</div>" if RUN_MODE in ("v3","v4","v5","v6") else ""}
  {"<div class='hw-cgt'>CGT &nbsp;&nbsp;&nbsp;&nbsp;✓ init</div>" if RUN_MODE in ("v4","v5","v6") else ""}
  {"<div class='hw-gw'>DGW &nbsp;&nbsp;&nbsp;&nbsp;" + f"{float(dgw_last):.4f}</div>" if RUN_MODE=="v6" else ""}
</div>

<div class="hw-snap">{snap_status}</div>

<div class="hw-legend">
  <span style="color:#ffb84f;">●</span> Sublattice A — tokens frequentes<br>
  <span style="color:#4fb0ff;">●</span> Sublattice B — tokens raros<br>
  <span style="color:#ff6bff;">━</span> Fases Kuramoto<br>
  <span style="color:#7ec8ff;">〜</span> Arcos binding (Δθ&lt;π/4)<br>
  <span style="color:#4f8fff;">↓</span> Teacher beam GPT-2<br>
  {"<span style='color:#4fffa0;'>◉</span> CGT semantic clusters<br>" if RUN_MODE in ("v4","v5","v6") else ""}
  {"<span style='color:#ff9f43;'>⌁</span> Persistence landscape edges" if RUN_MODE in ("v5","v6") else ""}
</div>

{"<div class='hw-gamma-badge'>Γ=" + f"{gamma:.2f}</div>" if RUN_MODE in ("v3","v4","v5","v6") else ""}
{"<div class='hw-dgw-badge'>DGW=" + f"{float(dgw_last):.3f}</div>" if RUN_MODE=="v6" else ""}
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>
(function() {{
  const wrap   = document.getElementById("hw-root");
  const canvas = document.getElementById("hw-c");
  const W = wrap.clientWidth, H = 720;

  const renderer = new THREE.WebGLRenderer({{ canvas, antialias:true, alpha:false }});
  renderer.setSize(W, H);
  renderer.setClearColor(0x02030a, 1);
  renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));

  const scene  = new THREE.Scene();
  scene.fog    = new THREE.FogExp2(0x02030a, 0.055);
  const camera = new THREE.PerspectiveCamera(46, W/H, 0.01, 200);
  camera.position.set(0, 3.5, 9);
  camera.lookAt(0, 0.5, 0);

  // ── Data ────────────────────────────────────────────────────────────
  const tokenData  = {tokens_json};
  const phaseData  = {phases_json};
  const trailData  = {trail_json};
  const gwData     = {gw_json};
  const IS_V3V4V5V6 = {is_v3v4v5v6};
  const IS_V4V5V6   = {is_v4v5v6};
  const IS_V6       = {is_v6_str};
  const GAMMA       = {gamma:.4f};
  const CUR_RAD     = {cur_rad:.4f};
  const CUR_RDC     = {cur_rdc:.4f};
  const PH_COLOR    = 0x{ph_color[1:]};
  const LOGIT_STD   = {cur_logstd:.3f};
  const hasSnap     = {'true' if has_snap else 'false'};

  // ── Hyperboloid grid ─────────────────────────────────────────────────
  for (let i = 0; i < 9; i++) {{
    const y = (i - 4) * 0.65;
    const r = Math.cosh(y * 0.45) * CUR_RAD;
    const g = new THREE.TorusGeometry(r, 0.007, 6, 96);
    const op = 0.06 + 0.04 * (1 - Math.abs(i-4)/4.5);
    scene.add(new THREE.Mesh(g,
      new THREE.MeshBasicMaterial({{color:0x0d1a3a, transparent:true, opacity:op, side:THREE.DoubleSide}})));
  }}
  for (let j = 0; j < 36; j++) {{
    const ang = (j/36) * Math.PI * 2;
    const pts = [];
    for (let t = -3.2; t <= 3.2; t += 0.055) {{
      const r = Math.cosh(t*0.48) * CUR_RAD;
      pts.push(new THREE.Vector3(Math.cos(ang)*r, t*0.52, Math.sin(ang)*r));
    }}
    scene.add(new THREE.Line(
      new THREE.BufferGeometry().setFromPoints(pts),
      new THREE.LineBasicMaterial({{color:0x060e1e, transparent:true, opacity:0.35}})));
  }}

  // ── Embedding shell ──────────────────────────────────────────────────
  const shellG = new THREE.TorusGeometry(CUR_RAD, 0.014, 8, 128);
  const shellM = new THREE.MeshBasicMaterial({{color:0xffb84f, transparent:true, opacity:0.65}});
  const shell  = new THREE.Mesh(shellG, shellM);
  shell.rotation.x = Math.PI/2;
  scene.add(shell);
  const glowG = new THREE.TorusGeometry(CUR_RAD, 0.22, 8, 128);
  const glowM = new THREE.MeshBasicMaterial({{color:0xffb84f, transparent:true, opacity:0.05}});
  const glow  = new THREE.Mesh(glowG, glowM);
  glow.rotation.x = Math.PI/2;
  scene.add(glow);

  // ── Phase protocol ring (v6): changes color by phase ────────────────
  if (IS_V6) {{
    const phRingG = new THREE.TorusGeometry(CUR_RAD * 1.08, 0.012, 6, 96);
    const phRingM = new THREE.MeshBasicMaterial({{
      color: PH_COLOR, transparent:true, opacity:0.5 }});
    const phRing  = new THREE.Mesh(phRingG, phRingM);
    phRing.rotation.x = Math.PI/2;
    phRing.position.y = -0.08;
    scene.add(phRing);
  }}

  // ── Token particles ──────────────────────────────────────────────────
  let animTokens = [];
  if (tokenData.length > 0) {{
    const freqT = tokenData.filter(t => t.sub==="A");
    const rareT = tokenData.filter(t => t.sub==="B");
    function makeCloud(tokens, color, size, opacity) {{
      const pos = new Float32Array(tokens.length*3);
      tokens.forEach((t,i) => {{ pos[i*3]=t.x; pos[i*3+1]=t.y; pos[i*3+2]=t.z; }});
      const g = new THREE.BufferGeometry();
      g.setAttribute("position", new THREE.BufferAttribute(pos,3));
      scene.add(new THREE.Points(g, new THREE.PointsMaterial({{
        color, size, transparent:true, opacity }})));
    }}
    makeCloud(freqT, 0xffb84f, 0.065, 0.9);
    makeCloud(rareT, 0x4fb0ff, 0.09,  0.85);

    // Semantic edges
    const MAX_D = 0.75;
    freqT.slice(0,100).forEach((a,i) => {{
      freqT.slice(i+1,100).forEach(b => {{
        const dx=a.x-b.x,dy=a.y-b.y,dz=a.z-b.z;
        const d=Math.sqrt(dx*dx+dy*dy+dz*dz);
        if (d<MAX_D) {{
          scene.add(new THREE.Line(
            new THREE.BufferGeometry().setFromPoints([
              new THREE.Vector3(a.x,a.y,a.z),
              new THREE.Vector3(b.x,b.y,b.z)]),
            new THREE.LineBasicMaterial({{
              color:0xffb84f, transparent:true, opacity:0.10*(1-d/MAX_D)}})));
        }}
      }});
    }});

    // Rare auras
    rareT.slice(0,15).forEach(t => {{
      const s = new THREE.Mesh(
        new THREE.SphereGeometry(0.07,6,6),
        new THREE.MeshBasicMaterial({{color:0x4fb0ff, transparent:true, opacity:0.11, wireframe:true}}));
      s.position.set(t.x,t.y,t.z);
      scene.add(s);
    }});

    // ── Temporal binding arcs ─────────────────────────────────────────
    if (IS_V3V4V5V6 && phaseData.length > 0) {{
      const THRESH = Math.PI / 4;
      let cnt = 0;
      for (let a = 0; a < Math.min(tokenData.length,80) && cnt<120; a++) {{
        for (let b = a+1; b < Math.min(tokenData.length,80) && cnt<120; b++) {{
          const pa = phaseData[a % phaseData.length];
          const pb = phaseData[b % phaseData.length];
          const dPh  = Math.abs(pa-pb)%(Math.PI*2);
          const dPhM = Math.min(dPh, Math.PI*2-dPh);
          if (dPhM < THRESH) {{
            const ta=tokenData[a], tb=tokenData[b];
            const dx=ta.x-tb.x,dy=ta.y-tb.y,dz=ta.z-tb.z;
            if (Math.sqrt(dx*dx+dy*dy+dz*dz) < 1.6) {{
              const mid = new THREE.Vector3(
                (ta.x+tb.x)*0.25, (ta.y+tb.y)*0.25+0.25, (ta.z+tb.z)*0.25);
              const curve = new THREE.QuadraticBezierCurve3(
                new THREE.Vector3(ta.x,ta.y,ta.z), mid,
                new THREE.Vector3(tb.x,tb.y,tb.z));
              const hue = (pa/(Math.PI*2))%1;
              const col = new THREE.Color().setHSL(hue,1.0,0.65);
              scene.add(new THREE.Line(
                new THREE.BufferGeometry().setFromPoints(curve.getPoints(14)),
                new THREE.LineBasicMaterial({{color:col,transparent:true,
                  opacity:0.32*(1-dPhM/THRESH)}})));
              cnt++;
            }}
          }}
        }}
      }}
    }}

    // ── CGT semantic clusters (v4/v5/v6) ─────────────────────────────
    if (IS_V4V5V6 && phaseData.length > 0 && tokenData.length > 10) {{
      const N_CL = 5;
      const clusters = Array.from({{length:N_CL}}, ()=>[]);
      tokenData.forEach((t,i) => {{
        const ph = phaseData[i % phaseData.length] || 0;
        const ci = Math.floor(((ph+Math.PI*2)%(Math.PI*2))/(Math.PI*2)*N_CL);
        clusters[ci].push(t);
      }});
      clusters.forEach((cl,ci) => {{
        if (cl.length < 3) return;
        const cx=cl.reduce((s,t)=>s+t.x,0)/cl.length;
        const cy=cl.reduce((s,t)=>s+t.y,0)/cl.length;
        const cz=cl.reduce((s,t)=>s+t.z,0)/cl.length;
        const col = new THREE.Color().setHSL(ci/N_CL*0.7, 0.8, 0.5);
        const s = new THREE.Mesh(
          new THREE.SphereGeometry(0.22,8,8),
          new THREE.MeshBasicMaterial({{color:col,transparent:true,opacity:0.07}}));
        s.position.set(cx,cy,cz);
        scene.add(s);
        // Cluster dot
        const dot = new THREE.Mesh(
          new THREE.SphereGeometry(0.04,5,5),
          new THREE.MeshBasicMaterial({{color:col,transparent:true,opacity:0.8}}));
        dot.position.set(cx,cy+0.28,cz);
        scene.add(dot);
      }});
    }}

    // ── Persistence landscape edges (v5/v6): critical topological edges
    if ((RUN_MODE==="v5"||IS_V6) && tokenData.length > 4) {{
      // Show "critical" edges — longest geodesic connections visible
      const sorted = tokenData.slice(0,50);
      let edgeCnt = 0;
      for (let a=0; a<sorted.length && edgeCnt<40; a++) {{
        for (let b=a+1; b<sorted.length && edgeCnt<40; b++) {{
          const ta=sorted[a], tb=sorted[b];
          const dx=ta.x-tb.x,dy=ta.y-tb.y,dz=ta.z-tb.z;
          const d=Math.sqrt(dx*dx+dy*dy+dz*dz);
          if (d > 0.9 && d < 1.8) {{
            // "critical edges" — topology-defining connections
            const col = new THREE.Color().setHSL(0.38, 0.9, 0.6);
            scene.add(new THREE.Line(
              new THREE.BufferGeometry().setFromPoints([
                new THREE.Vector3(ta.x,ta.y,ta.z),
                new THREE.Vector3(tb.x,tb.y,tb.z)]),
              new THREE.LineBasicMaterial({{color:col,transparent:true,opacity:0.18}})));
            edgeCnt++;
          }}
        }}
      }}
    }}

  }} else {{
    // Procedural fallback
    const N=550, pos=new Float32Array(N*3), col=new Float32Array(N*3);
    for (let i=0;i<N;i++) {{
      const theta=Math.random()*Math.PI*2;
      const phi=(Math.random()-0.5)*0.5;
      const freq=Math.random()<0.3;
      const r=CUR_RAD+(freq?Math.random()*0.14:1.5+Math.random()*2.8);
      pos[i*3]=Math.cos(theta)*r; pos[i*3+1]=phi*2.2; pos[i*3+2]=Math.sin(theta)*r;
      col[i*3]=freq?1.0:0.3; col[i*3+1]=freq?0.72:0.69; col[i*3+2]=freq?0.31:1.0;
    }}
    const g=new THREE.BufferGeometry();
    g.setAttribute("position",new THREE.BufferAttribute(pos,3));
    g.setAttribute("color",new THREE.BufferAttribute(col,3));
    scene.add(new THREE.Points(g, new THREE.PointsMaterial({{
      size:0.058, vertexColors:true, transparent:true, opacity:0.9 }})));
  }}

  // ── Kuramoto lines ───────────────────────────────────────────────────
  const kuLines = [];
  for (let h=0; h<8; h++) {{
    const baseA = (h/8)*Math.PI*2;
    const initP = phaseData.length>h ? phaseData[h] : Math.random()*Math.PI*2;
    const pts   = [new THREE.Vector3(0,0,0),
                   new THREE.Vector3(Math.cos(baseA)*CUR_RAD,0,Math.sin(baseA)*CUR_RAD)];
    const col   = new THREE.Color().setHSL(h/8*0.65+0.55, 0.9, 0.65);
    const line  = new THREE.Line(
      new THREE.BufferGeometry().setFromPoints(pts),
      new THREE.LineBasicMaterial({{color:col,transparent:true,opacity:0.65}}));
    scene.add(line);
    kuLines.push({{line, baseA, phase:initP, speed:0.011+Math.random()*0.008}});
  }}

  // Γ ring
  if (IS_V3V4V5V6 && GAMMA > 0.04) {{
    const gr = new THREE.TorusGeometry(CUR_RAD*(0.3+GAMMA*0.7), 0.022, 8, 64);
    const gm = new THREE.MeshBasicMaterial({{color:0xff6bff, transparent:true,
      opacity:0.4+GAMMA*0.4}});
    const gring = new THREE.Mesh(gr, gm);
    gring.rotation.x = Math.PI/2;
    gring.position.y = 0.38;
    scene.add(gring);
  }}

  // ── GW divergence visualisation (v6) ─────────────────────────────────
  // Spiral ribbon: DGW(t) mapped onto radius — phase transition becomes visible
  if (IS_V6 && gwData.length > 1) {{
    const gwPts = gwData.map((d,i) => {{
      const angle = (i/gwData.length) * Math.PI * 3;
      const r2    = CUR_RAD * (1.2 + d.dgw * 2.5);
      return new THREE.Vector3(
        Math.cos(angle)*r2, -1.6 + i/gwData.length*0.8,
        Math.sin(angle)*r2);
    }});
    scene.add(new THREE.Line(
      new THREE.BufferGeometry().setFromPoints(gwPts),
      new THREE.LineBasicMaterial({{color:0xff9f43,transparent:true,opacity:0.55}})));
  }}

  // ── Teacher beam ─────────────────────────────────────────────────────
  const beamG = new THREE.CylinderGeometry(0.01, 0.45, 5, 8, 1, true);
  const beamM = new THREE.MeshBasicMaterial({{color:0x4f8fff,transparent:true,
    opacity:0.10, side:THREE.DoubleSide}});
  const beam = new THREE.Mesh(beamG, beamM);
  beam.position.y = 2.5; scene.add(beam);

  // Distillation particles
  const ND=55, dp=new Float32Array(ND*3), dProg=[];
  for (let i=0;i<ND;i++) {{
    dp[i*3]=(Math.random()-0.5)*0.65;
    dp[i*3+1]=Math.random()*5-2.5;
    dp[i*3+2]=(Math.random()-0.5)*0.65;
    dProg.push(Math.random());
  }}
  const dGeo=new THREE.BufferGeometry();
  dGeo.setAttribute("position",new THREE.BufferAttribute(dp,3));
  scene.add(new THREE.Points(dGeo, new THREE.PointsMaterial({{
    color:0x4f8fff, size:0.033, transparent:true, opacity:0.55 }})));

  // PPL convergence ribbon
  if (trailData.length > 2) {{
    const rpts = trailData.map((d,i) => {{
      const a  = (i/trailData.length) * Math.PI * 4;
      const r2 = Math.max(0.2, Math.min(CUR_RAD*1.7, d.ppl/550));
      return new THREE.Vector3(Math.cos(a)*r2*2, -1.9+i/trailData.length*0.65, Math.sin(a)*r2*2);
    }});
    scene.add(new THREE.Line(
      new THREE.BufferGeometry().setFromPoints(rpts),
      new THREE.LineBasicMaterial({{color:0x0e1b40,transparent:true,opacity:0.55}})));
  }}

  // ── Lights ───────────────────────────────────────────────────────────
  scene.add(new THREE.AmbientLight(0x0d1525, 3));
  const pl1=new THREE.PointLight(0x4f8fff,2.2,16); pl1.position.set(0,4.5,0); scene.add(pl1);
  const pl2=new THREE.PointLight(0xffb84f,1.0,10); pl2.position.set(4,-1,3);  scene.add(pl2);
  if (IS_V3V4V5V6) {{
    const pl3=new THREE.PointLight(0xff6bff,0.9,9); pl3.position.set(-3,2,-3); scene.add(pl3);
  }}
  if (IS_V4V5V6) {{
    const pl4=new THREE.PointLight(0x7ec8ff,0.6,8); pl4.position.set(0,-2,4);  scene.add(pl4);
  }}
  if (IS_V6) {{
    const pl5=new THREE.PointLight(0xff9f43,0.5,7); pl5.position.set(3,3,-3);  scene.add(pl5);
  }}

  // ── Animate ──────────────────────────────────────────────────────────
  let t = 0;
  function animate() {{
    requestAnimationFrame(animate);
    t += 0.0055;

    // Camera orbit — slightly tilted for depth
    const orbitR = 9 + Math.sin(t*0.07)*0.6;
    camera.position.x = Math.sin(t*0.095) * orbitR;
    camera.position.z = Math.cos(t*0.095) * orbitR;
    camera.position.y = 3.2 + Math.sin(t*0.055)*1.1;
    camera.lookAt(0, 0.4, 0);

    // Shell pulse
    const pulse = 1 + Math.sin(t*2.1)*0.013;
    shell.scale.set(pulse,pulse,pulse); glow.scale.set(pulse,pulse,pulse);
    shellM.opacity = 0.48 + Math.sin(t*2.8)*0.18;

    // Kuramoto evolution with geodesic coupling influence
    const frustration = IS_V3V4V5V6 ? (0.3 + GAMMA*0.2) : 0.28;
    kuLines.forEach(kl => {{
      kl.phase += kl.speed + CUR_RDC * 0.004;
      const angle = kl.baseA + Math.sin(kl.phase) * frustration;
      const pa = kl.line.geometry.attributes.position;
      pa.setXYZ(1, Math.cos(angle)*CUR_RAD, Math.sin(kl.phase)*0.22, Math.sin(angle)*CUR_RAD);
      pa.needsUpdate = true;
    }});

    // Distillation rain
    const dpA = dGeo.attributes.position.array;
    for (let i=0;i<ND;i++) {{
      dProg[i]=(dProg[i]+0.0028)%1.0;
      dpA[i*3+1]=5-dProg[i]*8;
    }}
    dGeo.attributes.position.needsUpdate=true;
    beamM.opacity=0.05+Math.sin(t*1.7)*0.045;

    renderer.render(scene, camera);
  }}
  animate();
}})();
</script>
"""
    return HTML(html)

display(build_hyperboloid_viz())


In [ ]:
"""Cell 6 — Live Loop Paralelo (GIF + Phase + 3D + Charts)"""
import threading, time, base64
from IPython.display import clear_output, display, HTML, Image as IPImage
import os; os.system("pip install -q imageio[ffmpeg] imageio-ffmpeg pillow kaleido 2>/dev/null")
import imageio, numpy as _np
from PIL import Image as PILImage
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go

REFRESH       = 60
CAPTURE_EVERY = 120
GIF_EVERY     = 5
VIDEO_OUT     = TIMELAPSE_DIR / "timelapse_v6.mp4"
GIF_PREVIEW   = TIMELAPSE_DIR / "preview_v6.gif"

_stop     = threading.Event()
_frames   = [0]
_last_cap = [0.0]
_last_gif = [0]

def _capture():
    train = read_csv(TRAIN_CSV); val = read_csv(VAL_CSV)
    if not train: return None
    m = latest_metrics(train, val)
    b_rows = read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6") else []
    bm     = latest_binding(b_rows)
    gw_rows= read_csv(GW_CSV) if RUN_MODE=="v6" else []
    steps  = [int(r["step"])          for r in train]
    ppls   = [float(r["ppl"])         for r in train if r.get("ppl") and float(r["ppl"])<5000]
    rdcs   = [float(r["rdc_ema"])     for r in train if r.get("rdc_ema")]
    losses = [float(r["loss"])        for r in train if r.get("loss")]
    divs   = [float(r["diversity"])   for r in train if r.get("diversity")]
    vs     = [int(r["step"])          for r in val]
    vp     = [float(r["val_ppl"])     for r in val if r.get("val_ppl")]
    b_s    = [int(r["step"])               for r in b_rows]
    b_g    = [float(r.get("order_param",0) or 0) for r in b_rows]
    gw_s   = [int(r["step"])          for r in gw_rows]
    gw_d   = [float(r.get("dgw",0))  for r in gw_rows]

    n_rows = 4 if RUN_MODE=="v6" else (3 if RUN_MODE in ("v3","v4","v5") else 2)
    fig = make_subplots(rows=n_rows, cols=2, vertical_spacing=0.1, horizontal_spacing=0.08,
        subplot_titles=(["PPL","Loss","RDC","Diversity"] +
                       (["Γ","DGW"] if RUN_MODE=="v6" else [])+
                       (["phase",""] if RUN_MODE=="v6" else [])))
    kw=dict(mode="lines", showlegend=False)
    fig.add_trace(go.Scatter(x=steps[:len(ppls)],y=ppls, line=dict(color="#4f8fff",width=1.5),**kw),row=1,col=1)
    if vp: fig.add_trace(go.Scatter(x=vs[:len(vp)],y=vp, mode="lines+markers",
        line=dict(color="#4fffa0",width=2), marker=dict(size=3), showlegend=False), row=1,col=1)
    fig.add_trace(go.Scatter(x=steps[:len(losses)],y=losses, line=dict(color="#b07fff",width=1.5),**kw),row=1,col=2)
    fig.add_trace(go.Scatter(x=steps[:len(rdcs)],  y=rdcs,   line=dict(color="#ff6b6b",width=1.5),**kw),row=2,col=1)
    fig.add_hline(y=2.0, line_dash="dash", line_color="#ff6b6b", row=2, col=1)
    fig.add_trace(go.Scatter(x=steps[:len(divs)],  y=divs,   line=dict(color="#4fffa0",width=1.5),**kw),row=2,col=2)
    if n_rows >= 3 and b_s:
        fig.add_trace(go.Scatter(x=b_s,y=b_g, line=dict(color="#ff6bff",width=2),**kw),row=3,col=1)
        fig.add_hline(y=0.3, line_dash="dash", line_color="#ff6bff", row=3,col=1)
        if gw_d: fig.add_trace(go.Scatter(x=gw_s,y=gw_d, line=dict(color="#ff9f43",width=2),**kw),row=3,col=2)
    cur_ph = get_training_phase(m["step"])
    ph_str = f" P{(cur_ph or 0)+1}" if RUN_MODE=="v6" and cur_ph is not None else ""
    fig.update_layout(
        width=1280, height=700 if n_rows==4 else 560,
        paper_bgcolor="#05080f", plot_bgcolor="#0d1120",
        font=dict(color="#c8d4f0",size=9), showlegend=False,
        title=dict(
            text=f"{RUN_LABEL} · Step {m['step']:,} · PPL {m['ppl']:.1f} · Γ={bm['order_param']:.3f}{ph_str} · {time.strftime('%H:%M:%S')}",
            font=dict(size=10,color="#fff")),
        margin=dict(t=45,b=12,l=35,r=20))
    for i in range(1,n_rows+1):
        for j in range(1,3):
            fig.update_xaxes(gridcolor="#1a2035",row=i,col=j)
            fig.update_yaxes(gridcolor="#1a2035",row=i,col=j)
    try:
        png = pio.to_image(fig, format="png", width=1280, height=700 if n_rows==4 else 560)
        fp  = TIMELAPSE_DIR / f"frame_{m['step']:08d}.png"
        fp.write_bytes(png)
        return fp
    except Exception:
        return None

def _compile_gif(fps=3, max_frames=18):
    frames = sorted(TIMELAPSE_DIR.glob("frame_*.png"))[-max_frames:]
    if len(frames) < 2: return
    imgs = [_np.array(PILImage.open(f).convert("RGB").resize((854,532))) for f in frames]
    imageio.mimsave(str(GIF_PREVIEW), imgs, duration=1.0/fps, loop=0)

def _tl():
    time.sleep(5)
    while not _stop.is_set():
        if time.time()-_last_cap[0] >= CAPTURE_EVERY:
            try:
                if _capture(): _frames[0]+=1
                _last_cap[0]=time.time()
            except Exception as e: print(f"  ⚠️ {e}")
        time.sleep(5)

def _gif_w():
    time.sleep(30)
    while not _stop.is_set():
        if _frames[0]>=2 and _frames[0]-_last_gif[0]>=GIF_EVERY:
            try: _compile_gif(); _last_gif[0]=_frames[0]
            except: pass
        time.sleep(15)

_t1=threading.Thread(target=_tl,daemon=True); _t1.start()
_t2=threading.Thread(target=_gif_w,daemon=True); _t2.start()

print(f"🚀 {RUN_LABEL}")
print(f"   📸/{CAPTURE_EVERY}s  🎞️/{GIF_EVERY}fr  🖥️/{REFRESH}s")

try:
    while True:
        clear_output(wait=True)
        train=read_csv(TRAIN_CSV); val=read_csv(VAL_CSV)
        m=latest_metrics(train,val) if train else {}
        bm=latest_binding(read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6") else [])
        gw_rows=read_csv(GW_CSV) if RUN_MODE=="v6" else []
        gw_last=float(gw_rows[-1]["dgw"]) if gw_rows else 0.0
        cur_ph=get_training_phase(m.get("step",0))
        ph_disp=f" · {PHASE_LABELS['v6'][cur_ph]}" if cur_ph is not None and RUN_MODE=="v6" else ""
        nxt=max(0, int(CAPTURE_EVERY-(time.time()-_last_cap[0])))

        print(f"[{time.strftime('%H:%M:%S')}] step={m.get('step',0):,} "
              f"ppl={m.get('ppl',9999):.1f} rdc={m.get('rdc',0):.3f} "
              f"Γ={bm['order_param']:.3f} DGW={gw_last:.4f}{ph_disp} | "
              f"📸{_frames[0]} · next:{nxt}s")

        # GIF preview
        if GIF_PREVIEW.exists():
            gb = base64.b64encode(GIF_PREVIEW.read_bytes()).decode()
            gk = GIF_PREVIEW.stat().st_size/1024
            display(HTML(f'''<div style="text-align:center;margin:6px 0;">
              <div style="font-family:'DM Mono',monospace;font-size:9px;color:#3a4a6a;margin-bottom:3px;">
                🎞️ {_frames[0]} frames · {gk:.0f}KB</div>
              <img src="data:image/gif;base64,{gb}"
                style="width:100%;max-width:1000px;border-radius:6px;border:1px solid #1a2540;"/>
            </div>'''))

        render_phase_portrait()
        display(build_hyperboloid_viz())
        render_charts()
        print(f"\nPróximo refresh: {REFRESH}s")
        time.sleep(REFRESH)

except KeyboardInterrupt:
    print(f"\n⏹️  {_frames[0]} frames")
    _stop.set(); _t1.join(8); _t2.join(8)
    if _frames[0]>=2:
        try:
            w=imageio.get_writer(str(VIDEO_OUT),fps=4,codec="libx264",quality=8,pixelformat="yuv420p")
            for f in sorted(TIMELAPSE_DIR.glob("frame_*.png")):
                w.append_data(_np.array(PILImage.open(f).convert("RGB")))
            w.close()
            print(f"✅ {VIDEO_OUT}")
        except Exception as e: print(f"⚠️ {e}")
        _compile_gif(fps=4, max_frames=9999)
        print(f"✅ {GIF_PREVIEW}")


[06:09:30] step=500 ppl=1260.5 rdc=0.163 Γ=0.000 DGW=0.0000 · ALIGN | 📸0 · next:1s
⏳ Aguardando snapshot de binding...



Próximo refresh: 60s


In [ ]:
"""Cell 7 — Playback timelapse"""
from IPython.display import Video, display
if VIDEO_OUT.exists():
    print(f"✅ {VIDEO_OUT.stat().st_size/1e6:.1f}MB")
    display(Video(str(VIDEO_OUT), embed=True, width=1280))
else:
    frames = sorted(TIMELAPSE_DIR.glob("frame_*.png"))
    print(f"⏳ {len(frames)} frames — Ctrl+C na Cell 6 para compilar")
